# Les 13 - Agentgeheugen met Cognee Kennisgrafieken


## Setup

Deze notebook toont hoe je een intelligente **code-assistent** bouwt met persistente geheugen via [**Cognee**](https://www.cognee.ai/) kennissengrafen en het **Microsoft Agent Framework** (MAF).

Cognee zet ongestructureerde tekst om in een gestructureerde, doorzoekbare kennissengraaf ondersteund door vector-embeddding — waardoor je agent over een rijk, relatie-bewust langetermijngeheugen beschikt.

### Wat je leert
1. **Bouw Kennissengrafen**: Zet ontwikkelaarsprofielen en best practices om in gestructureerde, doorzoekbare kennis.
2. **Integreer Cognee met MAF**: Gebruik `@tool` functies om een MAF-agent Cognee’s kennissengraaf te laten doorzoeken.
3. **Sessiebewuste Gesprekken**: Behoud context over meerdere vragen in dezelfde sessie.
4. **Langetermijngeheugen**: Behoud belangrijke kennis over sessies heen en haal deze op in nieuwe gesprekken.

### Vereisten
- Python 3.9+
- Redis lokaal draaien (`docker run -d -p 6379:6379 redis`) voor sessiebeheer
- Een LLM API-sleutel (bijv. OpenAI) — zet `LLM_API_KEY` in `.env`
- `CACHING=true` in `.env` (vereist voor Cognee-sessies)
- Een Azure AI Foundry-project met een gedeployed chatmodel
- `AZURE_AI_PROJECT_ENDPOINT` en `AZURE_AI_MODEL_DEPLOYMENT_NAME` in `.env`
- Azure CLI ingelogd (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Typen Agentgeheugen

Deze notitieboek verkent dezelfde drie geheugentypen uit het hoofdnotitieboek van Les 13, maar gebruikt Cognee als de backend voor langetermijngeheugen:

| Geheugentype | Mechanisme | Levensduur |
|---|---|---|
| **Werkgeheugen** | `agent.create_session()` (MAF) | Enkele conversatie |
| **Kortetermijn** | Cognee sessie-cache (Redis) | Enkele sessie |
| **Langetermijn** | Cognee kennisgrafiek + vectoren | Permanent |

### Cognee's Geheugenarchitectuur
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Bereid Cognee-opslag voor


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Deel 1 — Het Kennisbestand Opbouwen

We verwerken drie soorten gegevens om een uitgebreide kennisbasis te creëren voor onze codeerassistent:

1. **Ontwikkelaarprofiel** — persoonlijke expertise en technische achtergrond
2. **Python Best Practices** — de Zen van Python met praktische richtlijnen
3. **Historische Gesprekken** — eerdere vraag- en antwoordsessies tussen ontwikkelaars en AI-assistenten


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Visualiseer de Kennisgrafiek

Cognee kan een interactieve HTML-visualisatie weergeven van de entiteiten en relaties die het heeft geëxtraheerd.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Verrijk Geheugen met Memify

`memify()` analyseert de kenniskgrafiek en genereert intelligente regels — die patronen, best practices en relaties tussen concepten identificeren.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Deel 2 — MAF-agent met Cognee Tools

Nu maken we een MAF-agent die de kennisgrafiek van Cognee kan bevragen via `@tool`-functies. Hiermee kan de agent de volledige kracht van grafiekbewuste semantische zoekopdrachten benutten, terwijl de conversatiecontext via sessies behouden blijft.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Werkgeheugen met Sessies

De `AgentSession` (gemaakt via `agent.create_session()`) biedt werkgeheugen binnen een sessie. De agent kan terugverwijzen naar eerdere berichten terwijl hij ook de lange-termijn kennisgrafiek van Cognee raadpleegt.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Nieuwe sessie — Langetermijngeheugen blijft behouden

Het starten van een nieuwe sessie wist het werkgeheugen, maar de Cognee kennisgrafiek is nog steeds beschikbaar. De agent kan dezelfde langetermijnkennis ophalen in een helemaal nieuw gesprek.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Samenvatting

In dit notebook heb je een codeerassistent gebouwd die **MAF's werkgeheugen** (`agent.create_session()`) combineert met **Cognee's langetermijn kennisgrafiek**.

### Wat je hebt geleerd
1. **Bouw van kennisgrafiek**: Cognee verwerkt ongestructureerde tekst en bouwt een grafiek + vectorgeheugen.
2. **Grafiek verrijking met memify**: Afgeleide feiten en rijkere relaties bovenop je bestaande grafiek.
3. **MAF + Cognee integratie**: `@tool` functies laten een MAF-agent op natuurlijke wijze Cognee's grafiek raadplegen.
4. **Werkgeheugen + langetermijngeheugen**: `AgentSession` (via `agent.create_session()`) biedt sessiecontext terwijl Cognee blijvende kennis levert.
5. **Gefilterd zoeken met NodeSets**: Specifieke subsets van de kennisgrafiek targeten (bijv. alleen principes).

### Belangrijkste inzichten
- **Cognee** verandert ruwe tekst in gestructureerd, relatiebewust geheugen — krachtiger dan een platte vectoropslag.
- **`@tool` functies** verbinden MAF-agenten en externe kennissystemen op een nette manier.
- **`AgentSession`** (via `agent.create_session()`) houdt context per conversatie gescheiden van langlevende kennis.
- Dezelfde kennisgrafiek dient meerdere sessies en agenten.

### Toepassingen in de praktijk
- **Ontwikkelaars copilots**: Code review, incidentanalyse, architectuurassistenten
- **Klantenservice copilots**: Supportagenten over productdocumentatie, FAQ's en CRM-notities
- **Interne expert copilots**: Beleids-, juridische, of beveiligingsassistenten die redeneren over richtlijnen
- **Geïntegreerde datalagen**: Gecombineerde gestructureerde en ongestructureerde data in één doorzoekbare grafiek

### Volgende stappen
- Experimenteren met temporeel bewustzijn in Cognee
- Definiëren van een OWL-ontologie voor domeinspecifieke grafiekkwaliteit
- Gebruikersfeedbackloops toevoegen om retrieval in de tijd te verbeteren
- Opschalen naar multi-agent systemen die dezelfde Cognee-geheugenlaag delen


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:  
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, kan het voorkomen dat geautomatiseerde vertalingen fouten of onnauwkeurigheden bevatten. Het originele document in de oorspronkelijke taal dient als de gezaghebbende bron te worden beschouwd. Voor cruciale informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor enige misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
